In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from T_method import LayeredStructure
from My_plotter import Style, Plotter
from scipy.optimize import minimize

In [ ]:
st = Style()
ndf = 150 # кол-во точек по df
min_df = -0.5
max_df = 0.5
df_width = (max_df - min_df)*100 # ширина по df в процентах
points_per_m_percent = lambda m: int(ndf * m / df_width) # кол-во точек на m процентов по df
df = np.linspace(min_df, max_df, ndf) # относительный сдвиг частоты
def sigma(x, thres=0, k=3):
    return 1/(1+np.exp(-k*(x - thres)))
def penalty(x, thres=0, k=3, c=3):
    return c/k * np.log(1 + np.exp(-k*(x - thres)))

In [ ]:

alpha1 = np.array([3, -1.2, 2.9, -1.2, 1])*1.4
beta1 = np.array([2.8, 0.75, 0.65, 0.6, 0.5])
structure1 = LayeredStructure(alpha1, beta=beta1)
dir1 = 10*np.log10(structure1.directivity(df))
dir1 = sigma(dir1, thres=15, k=3)
fig, ax = plt.subplots()
pl = Plotter(ax, st)
pl.plot(df*100, dir1, label='struc1')
pl.set_xlabel('df/f %')
pl.set_ylabel('Directivity (dB)')
pl.set_title('Directivity vs Frequency Offset')
pl.set_ylim((0, 1))
pl.finalize()
ax.axhline(18, color='gray', linestyle='--', alpha=0.5)
plt.show()


In [ ]:
count = 0
def objective_function(params):
    global count
    n = len(params) // 2
    m_target = points_per_m_percent(25)  # 10% of df width
    alpha = np.array(params[:n])
    beta = np.array(params[n:])
    structure = LayeredStructure(alpha, beta=beta)
    directivity = 10*np.log10(structure.directivity(df))
    target_f = -np.inf
    for i in range(0, ndf-m_target):
        segment_i = directivity[i:i+m_target]
        target_f_i = np.sum(sigma(segment_i, thres=15, k=3) - penalty(segment_i, thres=15, k=3, c=0.1))
        if target_f_i > target_f:
            target_f = target_f_i
    print(f"Iteration {count}: target_f = {target_f/ndf*df_width:.4f}")
    count += 1
    return -target_f  # We want to maximize target_f, so we minimize the negative of it
bounds = [
    (0,  7.0), (-6.0, 0), (0, 6.0), (-6.0, 0), (0, 6.0),  # alpha
    (0.1,  7.0), ( 0.1, 1.0), ( 0.1, 1.0), ( 0.1, 1.0), ( 0.1, 1.0),  # beta
]
alpha0 = np.array([3, -1.2, 2.9, -1.2, 1])*1.4
beta0 = np.array([2.8, 0.75, 0.65, 0.6, 0.5])
initial_params = np.concatenate([alpha0, beta0])
print(objective_function(initial_params)/ndf*df_width)  # Check initial score


In [ ]:
count = 0
res = minimize(
    objective_function,
    x0=initial_params,
    method="Powell",
    bounds=bounds,  # SciPy will keep it within bounds for Powell
    options=dict(
        maxfev=1000,    # iterations of Powell outer loop
        xtol=1e-3,      # param tolerance
        ftol=1e-4,      # objective tolerance
        disp=True
    )
)
best_params = res.x
best_score = -res.fun  # because we minimized -J

print("Success:", res.success)
print("Message:", res.message)
print("Best score (sum of sigmoids):", best_score/ndf*df_width)
print("Best params:", best_params)


In [ ]:
alph_opt = np.array(best_params[:5])
beta_opt = np.array(best_params[5:])
alph_0 = np.array([3, -1.2, 2.9, -1.2, 1])*1.4
beta_0 = np.array([2.8, 0.75, 0.65, 0.6, 0.5])
structure_opt = LayeredStructure(alph_opt, beta=beta_opt)
structure_0 = LayeredStructure(alph_0, beta_0)
dir_prev = 10*np.log10(structure_0.directivity(df))
dir1 = 10*np.log10(structure_opt.directivity(df))
fig, ax = plt.subplots()
pl = Plotter(ax, st)
pl.plot(df*100, dir1, label='struct_0')
pl.plot(df*100, dir_prev, label='struct_opt')
pl.set_xlabel('df/f %')
pl.set_ylabel('Directivity (dB)')
pl.set_title('Directivity vs Frequency Offset')
pl.set_ylim((0, 25))
pl.finalize()
ax.axhline(18, color='gray', linestyle='--', alpha=0.5)
ax.axhline(16, color='gray', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
print("Optimized alpha:", repr(best_params[:5]))
print("Optimized beta:", repr(best_params[5:]))